# 面试问题：Tool-use SFT 轨迹、序列化与 Loss Mask 怎样设计？

可直接复述的回答：Tool-use SFT 的训练单位是完整事件轨迹，而不是散落的文本片段。每个 tool call 要有 call ID、名称和严格参数，tool result 必须引用同一 ID。序列化时区分 system、user、assistant thought/call、tool result 和 final answer。通常监督 assistant 的 call 与 final answer，不把外部 tool result 当成模型要生成的标签。截断不能拆开 call-result 原子对，也不能把坏轨迹混进训练。执行式评测要真正运行工具 sandbox 并核对最终状态。模板、schema 和 mask 必须版本一致。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：订单工具轨迹与输入预览

五条脱敏轨迹覆盖查物流、退款、地址修改、库存和破损售后。每条含 user、call、result、final，保留 call ID 和参数，结构与真实 Agent 日志一致。


In [1]:
trajectories10 = [  # 构造五条完整工具使用轨迹。
    {"id": "t1", "user": "查询订单o-1物流", "call": {"id": "c1", "name": "track_order", "args": {"order_id": "o-1"}}, "result": {"call_id": "c1", "status": "in_transit"}, "final": "订单正在运输中"},  # 物流查询轨迹。
    {"id": "t2", "user": "为订单o-2申请退款", "call": {"id": "c2", "name": "create_refund", "args": {"order_id": "o-2"}}, "result": {"call_id": "c2", "status": "submitted"}, "final": "退款申请已提交"},  # 退款动作轨迹。
    {"id": "t3", "user": "修改订单o-3地址", "call": {"id": "c3", "name": "change_address", "args": {"order_id": "o-3"}}, "result": {"call_id": "c3", "status": "needs_confirmation"}, "final": "请确认新地址后再提交"},  # 需要确认的地址变更。
    {"id": "t4", "user": "检查商品s-8库存", "call": {"id": "c4", "name": "check_stock", "args": {"sku": "s-8"}}, "result": {"call_id": "c4", "status": "available"}, "final": "商品当前有库存"},  # 库存查询轨迹。
    {"id": "t5", "user": "商品破损需要售后", "call": {"id": "c5", "name": "open_claim", "args": {"reason": "damaged"}}, "result": {"call_id": "c5", "status": "photo_required"}, "final": "请上传破损照片"},  # 破损售后轨迹。
]  # 完成五条真实语义工具轨迹。
print("教学实验输入：trajectory | user | call | result | final")  # 输出轨迹预览表头。
for trajectory10 in trajectories10:  # 逐条展示完整事件链。
    print(trajectory10)  # 输出一条工具调用轨迹。


教学实验输入：trajectory | user | call | result | final
{'id': 't1', 'user': '查询订单o-1物流', 'call': {'id': 'c1', 'name': 'track_order', 'args': {'order_id': 'o-1'}}, 'result': {'call_id': 'c1', 'status': 'in_transit'}, 'final': '订单正在运输中'}
{'id': 't2', 'user': '为订单o-2申请退款', 'call': {'id': 'c2', 'name': 'create_refund', 'args': {'order_id': 'o-2'}}, 'result': {'call_id': 'c2', 'status': 'submitted'}, 'final': '退款申请已提交'}
{'id': 't3', 'user': '修改订单o-3地址', 'call': {'id': 'c3', 'name': 'change_address', 'args': {'order_id': 'o-3'}}, 'result': {'call_id': 'c3', 'status': 'needs_confirmation'}, 'final': '请确认新地址后再提交'}
{'id': 't4', 'user': '检查商品s-8库存', 'call': {'id': 'c4', 'name': 'check_stock', 'args': {'sku': 's-8'}}, 'result': {'call_id': 'c4', 'status': 'available'}, 'final': '商品当前有库存'}
{'id': 't5', 'user': '商品破损需要售后', 'call': {'id': 'c5', 'name': 'open_claim', 'args': {'reason': 'damaged'}}, 'result': {'call_id': 'c5', 'status': 'photo_required'}, 'final': '请上传破损照片'}


## 2. Baseline（基线）：所有事件文本都进入 Loss

朴素序列化把 user 和 tool result 也设为监督目标，模型会被训练去伪造外部观察。下面展示首条轨迹中错误的全监督标签。


In [2]:
def serialize10(trajectory10):  # 把结构化轨迹转换为事件 token。
    return [f"<user>{trajectory10['user']}", f"<assistant_call>{trajectory10['call']}", f"<tool_result>{trajectory10['result']}", f"<assistant_final>{trajectory10['final']}"]  # 保留四类事件边界。
serialized10 = [serialize10(trajectory10) for trajectory10 in trajectories10]  # 序列化全部工具轨迹。
baseline_labels10 = [[event10 for event10 in sequence10] for sequence10 in serialized10]  # 错误地监督所有事件。
print("基线首条事件与标签")  # 输出全监督基线表头。
for event10, label10 in zip(serialized10[0], baseline_labels10[0]):  # 逐事件展示错误标签。
    print(event10, "=>", label10)  # 显示 tool result 被当作模型目标。


基线首条事件与标签
<user>查询订单o-1物流 => <user>查询订单o-1物流
<assistant_call>{'id': 'c1', 'name': 'track_order', 'args': {'order_id': 'o-1'}} => <assistant_call>{'id': 'c1', 'name': 'track_order', 'args': {'order_id': 'o-1'}}
<tool_result>{'call_id': 'c1', 'status': 'in_transit'} => <tool_result>{'call_id': 'c1', 'status': 'in_transit'}
<assistant_final>订单正在运输中 => <assistant_final>订单正在运输中


## 3. 核心实现：事件校验与 Assistant-only Mask

先验证 call/result ID、工具名和参数字段，再只监督 `<assistant_call>` 与 `<assistant_final>`。坏轨迹进入隔离队列，不参与训练。


In [3]:
tool_schemas10 = {"track_order": {"order_id"}, "create_refund": {"order_id"}, "change_address": {"order_id"}, "check_stock": {"sku"}, "open_claim": {"reason"}}  # 定义五个工具允许参数。
def validate_trajectory10(trajectory10):  # 校验工具轨迹事件合同。
    call10 = trajectory10["call"]  # 读取 assistant tool call。
    result10 = trajectory10["result"]  # 读取外部工具结果。
    if call10["name"] not in tool_schemas10:  # 检查工具名是否冻结。
        return False, "unknown_tool"  # 拒绝未知工具。
    if set(call10["args"]) != tool_schemas10[call10["name"]]:  # 检查参数字段严格匹配 schema。
        return False, "schema_mismatch"  # 拒绝缺失或额外参数。
    if result10["call_id"] != call10["id"]:  # 检查结果是否引用同一调用。
        return False, "call_result_mismatch"  # 拒绝串线观察。
    return True, "valid"  # 接受完整原子轨迹。
masked_records10 = []  # 收集校验通过的事件和 loss mask。
validation_rows10 = []  # 保存逐轨迹校验结果。
for trajectory10, sequence10 in zip(trajectories10, serialized10):  # 逐条校验并构造监督标签。
    valid10, reason10 = validate_trajectory10(trajectory10)  # 执行 call-result 合同检查。
    mask10 = [False, True, False, True] if valid10 else [False] * 4  # 只监督 assistant call 与 final。
    validation_rows10.append((trajectory10["id"], valid10, reason10))  # 保存轨迹校验结果。
    if valid10:  # 只把合法轨迹加入训练数据。
        masked_records10.append((sequence10, mask10))  # 保存事件序列和监督 mask。
print("核心过程：trajectory | valid | reason")  # 输出轨迹验证表头。
for row10 in validation_rows10:  # 逐条展示合同结果。
    print(row10)  # 输出一条轨迹校验记录。
print("首条Assistant-only Mask")  # 输出事件监督对照表头。
for event10, supervised10 in zip(masked_records10[0][0], masked_records10[0][1]):  # 逐事件展示 loss mask。
    print(event10, "supervised=", supervised10)  # 显示外部结果不进入生成目标。


核心过程：trajectory | valid | reason
('t1', True, 'valid')
('t2', True, 'valid')
('t3', True, 'valid')
('t4', True, 'valid')
('t5', True, 'valid')
首条Assistant-only Mask
<user>查询订单o-1物流 supervised= False
<assistant_call>{'id': 'c1', 'name': 'track_order', 'args': {'order_id': 'o-1'}} supervised= True
<tool_result>{'call_id': 'c1', 'status': 'in_transit'} supervised= False
<assistant_final>订单正在运输中 supervised= True


## 4. 结果表、原子截断与结果解读

预算按事件而不是字符截断。必须保留 `assistant_call + tool_result` 原子对；预算不足时整对移除，并保留 user/final 或拒绝该样本。


In [4]:
event_costs10 = [18, 42, 28, 20]  # 设置首条轨迹四个事件的教学 token 成本。
budget10 = 75  # 设置小于完整轨迹总长的截断预算。
naive_kept10 = []  # 收集逐事件贪心截断结果。
used10 = 0  # 初始化朴素截断已用 token。
for event10, cost10 in zip(serialized10[0], event_costs10):  # 按事件顺序尝试装入预算。
    if used10 + cost10 <= budget10:  # 检查当前事件能否单独装入。
        naive_kept10.append(event10)  # 保留当前事件。
        used10 += cost10  # 累加 token 使用量。
atomic_groups10 = [([serialized10[0][0]], event_costs10[0]), (serialized10[0][1:3], event_costs10[1] + event_costs10[2]), ([serialized10[0][3]], event_costs10[3])]  # 把 call-result 绑定成原子组。
atomic_kept10 = []  # 收集原子截断结果。
atomic_used10 = 0  # 初始化原子截断 token 使用量。
for group10, cost10 in atomic_groups10:  # 按原子事件组尝试装入。
    if atomic_used10 + cost10 <= budget10:  # 检查整个组是否能容纳。
        atomic_kept10.extend(group10)  # 整组保留而不拆分。
        atomic_used10 += cost10  # 累加整组成本。
print("方法 | 保留事件 | token | call-result完整")  # 输出截断对照表头。
print("naive_event", naive_kept10, used10, all(any("<assistant_call>" in event10 for event10 in naive_kept10) == any("<tool_result>" in event10 for event10 in naive_kept10) for _ in [0]))  # 展示逐事件截断可能留下孤立 call。
print("atomic_group", atomic_kept10, atomic_used10, all(any("<assistant_call>" in event10 for event10 in atomic_kept10) == any("<tool_result>" in event10 for event10 in atomic_kept10) for _ in [0]))  # 展示原子组保持一致。
print("结果解读：Tool result是外部观察不参与loss，call-result在数据结构和截断上都必须成对")  # 解释监督与原子性的不同职责。


方法 | 保留事件 | token | call-result完整
naive_event ['<user>查询订单o-1物流', "<assistant_call>{'id': 'c1', 'name': 'track_order', 'args': {'order_id': 'o-1'}}"] 60 False
atomic_group ['<user>查询订单o-1物流', '<assistant_final>订单正在运输中'] 38 True
结果解读：Tool result是外部观察不参与loss，call-result在数据结构和截断上都必须成对


## 5. 失败案例与修正：错误 Call ID 与孤立 Call

构造 `result.call_id=c999` 的坏轨迹，朴素字符串训练不会发现；合同校验应隔离它。逐事件截断还可能留下 call 无 result，原子组修正后整对删除。


In [5]:
bad10 = {"id": "bad", "user": "查询订单", "call": {"id": "c9", "name": "track_order", "args": {"order_id": "o-9"}}, "result": {"call_id": "c999", "status": "done"}, "final": "已完成"}  # 构造 call-result 串线轨迹。
bad_valid10, bad_reason10 = validate_trajectory10(bad10)  # 执行严格事件校验。
naive_has_orphan10 = any("<assistant_call>" in event10 for event10 in naive_kept10) and not any("<tool_result>" in event10 for event10 in naive_kept10)  # 检查朴素截断是否产生孤立 call。
atomic_has_orphan10 = any("<assistant_call>" in event10 for event10 in atomic_kept10) != any("<tool_result>" in event10 for event10 in atomic_kept10)  # 检查原子截断是否成对。
print("失败行为：坏轨迹字符串可序列化但合同结果", bad_valid10, bad_reason10)  # 展示格式合法不等于事件一致。
print("失败截断孤立call", naive_has_orphan10, naive_kept10)  # 展示逐事件截断破坏原子性。
print("修正截断孤立call", atomic_has_orphan10, atomic_kept10)  # 展示原子组修正结果。


失败行为：坏轨迹字符串可序列化但合同结果 False call_result_mismatch
失败截断孤立call True ['<user>查询订单o-1物流', "<assistant_call>{'id': 'c1', 'name': 'track_order', 'args': {'order_id': 'o-1'}}"]
修正截断孤立call False ['<user>查询订单o-1物流', '<assistant_final>订单正在运输中']


## 6. 生产边界与轨迹制品

真实 Tool-use SFT 还要保存工具 schema 版本、权限、错误码、重试、审批和权威状态。执行式评测必须在 sandbox 中真正调用工具 stub，而不是只检查文本相似度。


In [6]:
tool_sft_contract10 = {"template": "tool-chat-v5", "schemas": "tool-catalog-v8", "supervised_events": ["assistant_call", "assistant_final"], "atomic_group": ["call", "result"], "bad_trace": "quarantine", "eval": "sandbox_execution"}  # 定义工具轨迹训练合同。
print("Tool-use SFT 制品", tool_sft_contract10)  # 展示模板、schema、mask和评测语义。
print("生产替换点：真实tokenizer、schema版本、审批/权限事件、sandbox执行、状态回读和坏轨迹隔离")  # 说明字符串教学序列的边界。


Tool-use SFT 制品 {'template': 'tool-chat-v5', 'schemas': 'tool-catalog-v8', 'supervised_events': ['assistant_call', 'assistant_final'], 'atomic_group': ['call', 'result'], 'bad_trace': 'quarantine', 'eval': 'sandbox_execution'}
生产替换点：真实tokenizer、schema版本、审批/权限事件、sandbox执行、状态回读和坏轨迹隔离


## 7. 最小回归测试

断言保护案例规模、loss mask、事件一致和原子截断。


In [7]:
assert len(trajectories10) >= 5  # 保证工具训练案例覆盖多种业务动作。
assert len(masked_records10) == len(trajectories10)  # 保证五条合法轨迹都进入训练集。
assert masked_records10[0][1] == [False, True, False, True]  # 保证只监督 assistant call 与 final。
assert bad_valid10 is False and bad_reason10 == "call_result_mismatch"  # 保证串线结果被隔离。
assert naive_has_orphan10 is True and atomic_has_orphan10 is False  # 保证原子截断修正孤立 call。
print("最小回归测试通过：轨迹校验、Loss Mask和原子截断稳定")  # 显示 Tool-use SFT 关键性质已验证。


最小回归测试通过：轨迹校验、Loss Mask和原子截断稳定
